In [3]:
import math
import sys
import getopt
import random
import numpy as np

# config variables
pile_size = 10
whose_turn = "computer" # computer or human
opponent = "random"     # random or expert or learning
DEBUG = True
learning_episodes = 250


def expert_move(n):
    '''
    :param n: state
    :return: action (not action id)
    '''
    # TODO: implement your expert move here
    return random.randint(1,2)


class QLearning:
    def __init__(self):
        # rows are actions and columns are states
        self.Q_table = np.zeros((2, pile_size + 1))
        self.epsilon = 0.1
        self.alpha = 0.3

    def get_learning_action(self, state):
        '''
        :param state:
        :return: the actual action, not action id
        '''
        if state == 1:
            return 1
        # break ties randomly and select a random action id with probability epsilon
        a = random.randint(0,1)
        if self.epsilon < random.uniform(0,1) and self.Q_table[0, state] != self.Q_table[1, state]:
            # get an index of an action with the highest value
            a = np.argmax(self.Q_table[:, state])
        return a + 1

    def get_optimal_action(self, state):
        if state == 1:
            return 1
        # break ties randomly
        a = random.randint(0,1)
        if self.Q_table[0, state] != self.Q_table[1, state]:
            # get an index of an action with the highest value
            a = np.argmax(self.Q_table[:, state])
        return a + 1

    def q_learning_step(self, prev_state, prev_action, state, r):
        prev_action_id = prev_action - 1
        if DEBUG:
            print("Q-table indices: s={} a={} s'={} r={}".format(prev_state, prev_action_id, state, r))
            self.print_policy()
        if prev_action_id >= 0:
            best_action_id = self.get_optimal_action(state) - 1
            # do Q-learning
            self.Q_table[prev_action_id, prev_state] = self.Q_table[prev_action_id, prev_state] * (1 - self.alpha) \
                                                  + self.alpha * (r + self.Q_table[best_action_id, state])
        return self.get_learning_action(state)

    def print_policy(self):
        # TODO: implement this method
        print("Not implemented.")


def who_is_first():
    if random.choice([True, False]) is True:
        return "computer"
    else:
        return "human"


if __name__ == "__main__":
    opts, args = getopt.getopt(sys.argv[1:], "hg:o:n:", ["help", "goes-first=", "opponent=", "num-learning-episodes="])
    for opt, arg in opts:
        if opt == '-h':
            print(sys.argv[0] + ' --goes-first [human|computer] --opponent [random|learning|expert]'
                                ' --num-learning-episodes [number]')
            sys.exit(0)
        elif opt in ("-g", "--goes-first"):
            whose_turn = arg
        elif opt in ("-o", "--opponent"):
            opponent = arg
        elif opt in ("-n", "--num-learning-episodes"):
            learning_episodes = int(arg)

    if learning_episodes > 0 and not opponent == "learning":
        print("learning_episodes will be ignored because the opponent is not learning.")
        learning_episodes = 0

    # a new RL agent; it will learn from scratch
    rl_player = QLearning()

    episode = 0
    while True:
        if episode > 0:
            # after one episode, we randomise who goes first
            whose_turn = who_is_first()
        print("New game no {} starts. {} plays first. The opponent is {}.".format(episode+1, whose_turn, opponent))
        curr_pile_size = pile_size
        # previous pile size and action with respect to the computer player; needed for Q-learning
        prev_pile_size = -1
        prev_move = -1
        while True:
            print("-" * 20)
            print("Current pile has " + str(curr_pile_size) + " stones: " + "*" * curr_pile_size)
            if whose_turn == "human":
                if episode < learning_episodes:
                    # This is another learning episode. The human player is replaced by another policy
                    # for fast learning.
                    # (1) learning against a random player
                    move = random.randint(1, 2)
                    # (2) learning against an expert
                    # TODO: implement this
                    # (3) learning against itself
                    # TODO: implement this
                    if curr_pile_size == 1:
                        move = 1
                else:
                    move = -1
                    if curr_pile_size == 1:
                        print("Only action 1 is possible.")
                        move = 1
                        input("Please press ENTER:\n")
                    else:
                        while not move == 1 and not move == 2:
                            move = int(input("Please enter 1 or 2 and press ENTER:\n"))
                print("The {} player moves next and removes {}.".format(whose_turn, move))
                curr_pile_size -= move
                if curr_pile_size == 0:
                    if opponent == "learning":
                        # tell the learning agent that she lost after her last move
                        rl_player.q_learning_step(prev_pile_size, prev_move, 0, -1.0)
                    print("You won!")
                    break
            else:
                move = -1
                if opponent == "random":
                    move = random.randint(1, 2)
                    if curr_pile_size == 1:
                        move = 1
                elif opponent == "expert":
                    move = expert_move(curr_pile_size)
                    if curr_pile_size == 1:
                        move = 1
                elif opponent == "learning":
                    # here, the Q-values of the previous pile size are updated
                    # (before the most recent move of the opponent)
                    move = rl_player.q_learning_step(prev_pile_size, prev_move, curr_pile_size, 0.0)
                    prev_move = move
                else:
                    print("Wrong opponent. See the --opponent parameter.")
                    sys.exit(1)
                if (curr_pile_size == 1 and move != 1) or (move != 1 and move != 2):
                    print("Wrong action.")
                    sys.exit(1)
                print("The {} player moves next and removes {}.".format(whose_turn, move))
                prev_pile_size = curr_pile_size
                curr_pile_size -= move
                if curr_pile_size == 0:
                    if opponent == "learning":
                        # tell the learning agent that she won; the most recent pile size and action
                        # led to the winning state 0; that's where we give the reward of 1
                        rl_player.q_learning_step(move, move, 0, 1.0)
                    print("The {} player won!".format(opponent))
                    break
            if whose_turn == "human":
                whose_turn = "computer"
            else:
                whose_turn = "human"
        if opponent == "learning" and DEBUG is True:
            with np.printoptions(precision=3, suppress=True):
                print(rl_player.Q_table)
        if episode >= learning_episodes:
            again = input("Play again y/n and press ENTER:\n")
            if again == "n":
                break
        episode += 1

GetoptError: option -f not recognized